# TarlAI - AI-Powered Agricultural Assistant
### Gemma 4 Good Hackathon Submission

**Team:** Altug Ikiz, Arda Akinci, Simittin, Zeynep Yonca Ozel  
**Track:** Impact Track - Digital Equity & Inclusivity  
**Model:** Gemma 4 E4B-IT (Instruction Tuned)

---

**Pipeline:**  
`Photo Upload → Gemma 4 Vision (Diagnosis) → Function Calling (Weather + Treatment) → Turkish Recommendation`

## 1. Environment Setup

In [ ]:
!pip install --upgrade transformers accelerate -q

## 2. Imports & Model Loading

In [ ]:
from transformers import pipeline, GenerationConfig, AutoProcessor, AutoModelForMultimodalLM
from PIL import Image
from IPython.display import display
import json
import re

MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"
# For Hugging Face Hub: MODEL_PATH = "google/gemma-4-E4B-it"

print("Loading Gemma 4 E4B-IT...")
vqa_pipe = pipeline(
    task="image-text-to-text",
    model=MODEL_PATH,
    device_map="auto",
    dtype="auto"
)

config = GenerationConfig.from_pretrained(MODEL_PATH)
config.max_new_tokens = 1024
gen_kwargs = dict(generation_config=config)
print("Model ready!")

## 3. Tool Definitions

TarlAI uses two tools that Gemma 4 can invoke via **native function calling**:
- `get_weather()` - Returns local weather conditions for spray timing
- `get_treatment()` - Returns disease-specific treatment options

In [ ]:
def get_weather(location: str) -> dict:
    """
    Get current weather information for a given location.
    Args:
        location: City name, e.g. 'Antalya' or 'Istanbul'
    Returns:
        Weather data including temperature, humidity, rain probability
    """
    # TODO: Replace with OpenWeatherMap API in production
    db = {
        "antalya": {"temp": 26, "humidity": 72, "rain_pct": 15, "wind_kmh": 12, "condition": "Sunny"},
        "istanbul": {"temp": 18, "humidity": 65, "rain_pct": 40, "wind_kmh": 22, "condition": "Cloudy"},
        "izmir": {"temp": 24, "humidity": 58, "rain_pct": 10, "wind_kmh": 15, "condition": "Sunny"},
        "mugla": {"temp": 25, "humidity": 60, "rain_pct": 20, "wind_kmh": 14, "condition": "Partly Cloudy"},
        "burdur": {"temp": 22, "humidity": 50, "rain_pct": 5, "wind_kmh": 8, "condition": "Sunny"},
    }
    key = location.lower().strip()
    for city, data in db.items():
        if city in key:
            return {"location": location, **data}
    return {"location": location, "temp": 20, "humidity": 60, "rain_pct": 25, "wind_kmh": 10, "condition": "Unknown"}


def get_treatment(disease_name: str) -> dict:
    """
    Get treatment information for a plant disease.
    Args:
        disease_name: Name of the disease, e.g. 'early_blight', 'powdery_mildew', 'leaf_spot'
    Returns:
        Treatment details including chemical and organic options
    """
    db = {
        "leaf_spot": {"name": "Leaf Spot", "chemical": "Copper-based fungicide, Mancozeb", "organic": "Bordeaux mixture, Neem oil", "interval": "7-14 days"},
        "early_blight": {"name": "Early Blight", "chemical": "Mancozeb, Chlorothalonil", "organic": "Neem oil, copper sulfate", "interval": "7-10 days"},
        "late_blight": {"name": "Late Blight", "chemical": "Metalaxyl, Bordeaux mixture", "organic": "Bordeaux mixture", "interval": "5-7 days"},
        "powdery_mildew": {"name": "Powdery Mildew", "chemical": "Sulfur-based fungicide", "organic": "Milk-water mix 40%", "interval": "7 days"},
        "downy_mildew": {"name": "Downy Mildew", "chemical": "Metalaxyl, Fosetyl-Al", "organic": "Copper hydroxide", "interval": "7-10 days"},
    }
    key = disease_name.lower().replace(" ", "_").strip()
    for disease, data in db.items():
        if disease in key or key in disease:
            return data
    return {"name": disease_name, "info": "Not found - consult local agricultural office"}

print("Tools loaded: get_weather(), get_treatment()")

## 4. Native Function Calling Demo

Gemma 4 has built-in special tokens for tool use (`<|tool_call>`, `<|tool_result>`).  
Here we demonstrate the model **autonomously deciding** which tools to call:

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_PATH, dtype="auto", device_map="auto")
fc_config = GenerationConfig.from_pretrained(MODEL_PATH)
fc_config.max_new_tokens = 512

tools = [get_weather, get_treatment]

# ── Prompt V2: parameterized, explicitly requests both tools ──────────────────
# V1 (baseline):
# messages = [{"role": "user", "content": "I found leaf spot disease on my grape plants in Antalya. When should I spray considering the weather? Respond in Turkish."}]

def build_fc_prompt(plant: str, disease: str, location: str) -> str:
    return (
        f"I am a farmer. My {plant} plants in {location} have been diagnosed with {disease}.\n\n"
        f"Please:\n"
        f"1. Check the current weather conditions in {location}\n"
        f"2. Look up the treatment protocol for {disease}\n\n"
        f"Use the available tools to gather this information."
    )

messages = [{"role": "user", "content": build_fc_prompt("grape", "leaf_spot", "Antalya")}]

prompt = processor.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False)
inputs = processor(text=prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, generation_config=fc_config)
response = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

print("Gemma 4 Tool Calls:")
print(response)


## 5. Execute Tools & Get Final Response

Now we run the tools and feed results back to Gemma 4:

In [ ]:
weather_result = get_weather("Antalya")
treatment_result = get_treatment("leaf_spot")

print("Weather:", json.dumps(weather_result, indent=2))
print("Treatment:", json.dumps(treatment_result, indent=2))

# Feed tool results back using Gemma 4 special token format
manual_prompt = prompt.rstrip()
manual_prompt += (
    '<|tool_call>call:get_weather{location:<|"|>Antalya<|"|>}<tool_call|>'
    '<|tool_call>call:get_treatment{disease_name:<|"|>leaf_spot<|"|>}<tool_call|>\n'
    '<|turn>tool\n'
    '<|tool_result>result:get_weather' + json.dumps(weather_result) + '<tool_result|>'
    '<|tool_result>result:get_treatment' + json.dumps(treatment_result) + '<tool_result|>'
    '<turn|>\n<|turn>model\n'
)

inputs2 = processor(text=manual_prompt, return_tensors="pt").to("cuda")
fc_config.max_new_tokens = 1024
output2 = model.generate(**inputs2, generation_config=fc_config)
final_response = processor.decode(output2[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=False)

print("\n=== Gemma 4 Final Response ===")
print(final_response)

## 6. TarlAI Complete Pipeline

The `tarlai_analyze()` function combines all three stages into a single call:

In [ ]:
def tarlai_analyze(image_path, location="Antalya"):
    """
    Complete TarlAI pipeline: Image -> Diagnosis -> Tools -> Turkish Recommendation

    Args:
        image_path: Path to the plant image file
        location: City name for weather data (default: Antalya)
    """
    img = Image.open(image_path)
    display(img.resize((400, 300)))

    print(f"\n>> Location: {location}")
    print(">> Diagnosing with Gemma 4...\n")

    # ── Stage 1: Vision Diagnosis ─────────────────────────────────────────────
    # V1 (baseline):
    # DIAGNOSIS_PROMPT = (
    #     'You are an expert agricultural engineer. Analyze this plant image. '
    #     'Respond ONLY with JSON: {"plant": "name", "disease": "name", '
    #     '"severity": "mild/moderate/severe", "disease_key": "snake_case_name"}'
    # )
    #
    # V2: schema-explicit with examples and edge-case fallback (recommended)
    DIAGNOSIS_PROMPT = (
        "You are an expert plant pathologist and agricultural engineer.\n\n"
        "Analyze the plant image carefully. Identify the plant species, "
        "the disease or health condition, and its severity.\n\n"
        "Respond ONLY with a valid JSON object matching this exact schema:\n"
        '{"plant": "<plant species, e.g. tomato, grape, lemon>", '
        '"disease": "<disease name, e.g. early blight, powdery mildew>", '
        '"severity": "<one of: mild, moderate, severe>", '
        '"disease_key": "<snake_case id, e.g. early_blight, powdery_mildew>"}\n\n'
        "If the image is unclear or no disease is visible, use: "
        '{"plant": "<plant or unknown>", "disease": "healthy", '
        '"severity": "none", "disease_key": "healthy"}\n\n'
        "Output JSON only. No explanation. No markdown."
    )

    msg1 = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": DIAGNOSIS_PROMPT}
    ]}]

    diag = vqa_pipe(msg1, return_full_text=False, generate_kwargs=gen_kwargs)
    diag_text = diag[0]['generated_text']

    try:
        # Handle both inline JSON and markdown code-fenced JSON
        match = re.search(r'```(?:json)?\s*(\{[^`]+\})\s*```', diag_text, re.DOTALL)
        if not match:
            match = re.search(r'\{[^{}]*\}', diag_text, re.DOTALL)
        if match:
            group = match.group(1) if match.lastindex else match.group()
            diagnosis = json.loads(group)
        else:
            diagnosis = {"plant": "Unknown", "disease": "Unknown",
                         "severity": "unknown", "disease_key": "unknown"}
    except Exception:
        diagnosis = {"plant": "Unknown", "disease": "Unknown",
                     "severity": "unknown", "disease_key": "unknown"}

    # ── Stage 2: Agentic tool use ─────────────────────────────────────────────
    weather = get_weather(location)
    treatment = get_treatment(diagnosis.get("disease_key", "unknown"))

    print(f">> Plant: {diagnosis.get('plant')}")
    print(f">> Disease: {diagnosis.get('disease')}")
    print(f">> Severity: {diagnosis.get('severity')}")
    print(f">> Weather: {weather.get('condition')}, {weather.get('temp')}C")
    print("\n>> Generating recommendation...\n")

    # ── Stage 3: Turkish Recommendation ──────────────────────────────────────
    # V1 (baseline): simple list, language mixed in, no severity awareness
    # V2: severity-aware urgency + structured sections (recommended)
    severity = diagnosis.get("severity", "moderate")
    urgency_map = {
        "mild":     "Durum henüz erken evrede, ama vakit kaybetmeden harekete geçmek önemli.",
        "moderate": "Durum orta şiddette, bu hafta ilaçlama başlamalısın.",
        "severe":   "ACELE ET! Hastalık ileri seviyede, bugün veya yarın ilaçlamalısın."
    }
    urgency = urgency_map.get(severity, urgency_map["moderate"])

    RECOMMENDATION_PROMPT = (
        "Sen TarlAI'sin — Türk çiftçileri için yapay zeka destekli tarım danışmanı.\n"
        "Aşağıdaki verilere dayanarak kapsamlı bir Türkçe tavsiye oluştur.\n"
        "Dili sade tut: lise mezunu bir çiftçinin anlayabileceği şekilde yaz.\n\n"
        f"ACİLİYET NOTU: {urgency}\n\n"
        f"VERİLER:\n"
        f"- Teşhis: {json.dumps(diagnosis, ensure_ascii=False)}\n"
        f"- Hava durumu ({weather.get('location', '')}): {weather.get('temp')}°C, "
        f"%{weather.get('humidity')} nem, {weather.get('condition')}, "
        f"rüzgar {weather.get('wind_kmh')} km/s\n"
        f"- Tedavi bilgisi: {json.dumps(treatment, ensure_ascii=False)}\n\n"
        "YAPI (tam olarak bu bölümleri kullan):\n\n"
        "## 1. Tespit\n"
        "[Hastalık adını ve şiddetini açıkla. Belirtileri kısaca tanımla.]\n\n"
        "## 2. Tedavi Seçenekleri\n"
        "**Kimyasal:** [İlaç adları ve uygulama sıklığı]\n"
        "**Organik/Doğal:** [Alternatif yöntemler]\n\n"
        "## 3. Spreylama Zamanlaması\n"
        "[Hava durumuna göre ne zaman ilaçlayacağını söyle — "
        "sabah/akşam, yağmur öncesi/sonrası, rüzgar durumu]\n\n"
        "## 4. Önleme\n"
        "[Bir sonraki sezonda bu hastalıktan nasıl korunur?]\n\n"
        "Sonunda tek cümlelik öz bir özet yaz. Samimi ve cesaretlendirici bir ton kullan."
    )

    msg2 = [{"role": "user", "content": [
        {"type": "text", "text": RECOMMENDATION_PROMPT}
    ]}]

    final = vqa_pipe(msg2, return_full_text=False, generate_kwargs=gen_kwargs)
    result = final[0]['generated_text'].replace("<turn|>", "").replace("<eos>", "").strip()

    print("=" * 50)
    print("TARLAI - ANALYSIS REPORT")
    print("=" * 50)
    print(result)
    print("=" * 50)

print("TarlAI pipeline ready!")


## 7. Demo

Run TarlAI on a sample diseased plant image:

In [ ]:
# Replace with your image path
tarlai_analyze("/kaggle/input/datasets/altugikiz/hastabitki/hastabitki.jpg", "Antalya")